# Self-Correcting Code Agent — Core Loop Prototype

Prototype of the ReAct + Reflexion loop described in `ReAct_Reflexion_Project_Overview.md`.

**What's implemented here:**
- Reasoning/Planning turn via a **Groq-hosted Qwen model**, configured through an env var (`MODEL_ID`) rather than hardcoded, since the exact model slug is expected to change
- A **`subprocess`-based execution sandbox** (real process isolation, not a restricted `exec()` namespace)
- Reflexion turn on failure, appended to the same conversation thread
- A **3-attempt cap** and a **human-in-the-loop confirmation** checkpoint after any successful run
- A trajectory logger for the demo/report
- A **vanilla HTML/CSS/JS front-end** (no UI framework) that lives in the repo's `frontend/` folder — served by a small Flask backend and exposed from Colab via a **Cloudflare quick tunnel**. Task box, live NDJSON-streamed trajectory panel, iteration counter, confirm/reject buttons.

Run the cells top to bottom. You'll be prompted for an API key in Section 2.

## 1. Install dependencies

In [ ]:
!pip install -q openai flask

## 2. Configure the LLM backend

Uses an OpenAI-compatible client pointed at **Groq**. The model id is read from the `MODEL_ID` env var — **not hardcoded** — since the team expects to swap the exact model slug over time. Current choice: `qwen/qwen3.8-27b`.

To change it later, either set the env var before running this cell (`os.environ["MODEL_ID"] = "..."`) or edit the default below — no other cell needs to change. Double-check the slug against Groq's current catalog at [console.groq.com/docs/models](https://console.groq.com/docs/models) before running, since hosted model names do change.

In [ ]:
import os

api_key = os.environ.get("GROQ_API_KEY")
if not api_key:
    try:
        from google.colab import userdata  # type: ignore
        api_key = userdata.get("GROQ_API_KEY")
    except Exception:
        pass
if not api_key:
    import getpass
    api_key = getpass.getpass("Enter your Groq API key (console.groq.com/keys): ")

# MODEL_ID comes from the environment, not a hardcoded literal in the API call below.
# setdefault() only fills it in if nothing else (shell env, Colab secret, prior cell) already set it.
os.environ.setdefault("MODEL_ID", "qwen/qwen3.8-27b")
MODEL_ID = os.environ["MODEL_ID"]

from openai import OpenAI

BASE_URL = "https://api.groq.com/openai/v1"
client = OpenAI(base_url=BASE_URL, api_key=api_key)

def call_llm(messages, temperature=0.2):
    """Single call in the task's ongoing thread -- used for both planning and reflection turns."""
    resp = client.chat.completions.create(
        model=MODEL_ID,
        messages=messages,
        temperature=temperature,
    )
    return resp.choices[0].message.content

## 3. System prompt

Defines the ReAct output format: a short *Thought*, then exactly one fenced Python code block. The code must be a complete, runnable script — whatever it prints is what becomes the Observation.

In [ ]:
SYSTEM_PROMPT = """You are a careful Python coding agent working inside a ReAct (Reason+Act) loop.

For every turn:
1. Write a short \"Thought:\" section explaining your plan or, on a retry, how you're addressing the previous failure/reflection.
2. Then write exactly ONE fenced python code block containing a complete, runnable script that accomplishes the task.
   - The script must PRINT whatever output demonstrates the result (stdout is the only thing the agent observes).
   - Do not use input() or any interactive prompts.
   - Do not access the network or the filesystem outside the current working directory.
3. Do not include more than one code block.

You will be shown the Observation (stdout/stderr/traceback) after each attempt, and may be asked to reflect on a failure before your next attempt."""

REFLECTION_PROMPT = (
    "That attempt did not solve the task. Diagnose the root cause of the failure as specifically as you can "
    "(not just what error occurred, but why the code produced it), and state concretely what you will change "
    "in the next attempt. Do not write code yet -- just the diagnosis and plan."
)

REJECTED_BY_USER_PROMPT = (
    "The code ran without error, but the user reviewed the output and confirmed it does NOT correctly solve the task. "
    + REFLECTION_PROMPT
)

## 4. Code extractor

Pulls the fenced code block out of the model's response and validates it parses before we ever try to run it.

In [ ]:
import re
import ast

CODE_BLOCK_RE = re.compile(r"```(?:python)?\s*\n(.*?)```", re.DOTALL)

def extract_code(response_text):
    """Returns (code, error). Exactly one of the two is None."""
    match = CODE_BLOCK_RE.search(response_text)
    if not match:
        return None, "No fenced python code block found in the response."
    code_str = match.group(1).strip()
    try:
        ast.parse(code_str)
    except SyntaxError as e:
        return None, f"Extracted code has a syntax error: {e}"
    return code_str, None

def extract_thought(response_text):
    idx = response_text.find("```")
    thought = response_text[:idx].strip() if idx != -1 else response_text.strip()
    return thought or "(no explicit reasoning provided)"

## 5. Execution sandbox (`subprocess`-based)

Each attempt runs as a **separate OS process** in its own interpreter (`sys.executable -I`), not `exec()` in the orchestrator's namespace. That's a real isolation boundary: the attempt cannot see or mutate the orchestrator's memory, and a fresh process per attempt means no state can leak between attempts.

**Documented limitation (matches §8/§11 of the design doc):** this isolates the *process*, but does not fully sandbox *system calls* — e.g. it doesn't block outbound network access or restrict which stdlib modules can be imported. A production version would add OS-level controls (containers, seccomp, `resource.setrlimit`, an egress-blocking network namespace). For this prototype we rely on: a fresh scratch directory per attempt, a timeout, a minimal environment, and truncated output capture.

In [ ]:
import subprocess
import sys
import tempfile
import os
import shutil

def run_in_sandbox(code_str, timeout=10, max_output_chars=4000):
    scratch_dir = tempfile.mkdtemp(prefix="agent_attempt_")
    script_path = os.path.join(scratch_dir, "attempt.py")
    with open(script_path, "w", encoding="utf-8") as f:
        f.write(code_str)

    env = {"PATH": os.environ.get("PATH", ""), "PYTHONIOENCODING": "utf-8"}

    timed_out = False
    try:
        proc = subprocess.run(
            [sys.executable, "-I", script_path],
            cwd=scratch_dir,
            capture_output=True,
            text=True,
            timeout=timeout,
            env=env,
        )
        stdout, stderr, returncode = proc.stdout, proc.stderr, proc.returncode
    except subprocess.TimeoutExpired as e:
        timed_out = True
        stdout = e.stdout or ""
        stderr = (e.stderr or "") + f"\n[Timed out after {timeout}s]"
        returncode = None
    finally:
        shutil.rmtree(scratch_dir, ignore_errors=True)

    def trunc(s):
        return s if len(s) <= max_output_chars else s[:max_output_chars] + "\n...[truncated]"

    success = (not timed_out) and returncode == 0
    return {
        "success": success,
        "stdout": trunc(stdout),
        "stderr": trunc(stderr),
        "returncode": returncode,
        "timed_out": timed_out,
    }

def format_observation(result):
    lines = []
    if result["timed_out"]:
        lines.append("Execution timed out.")
    lines.append(f"Return code: {result['returncode']}")
    if result["stdout"]:
        lines.append("stdout:\n" + result["stdout"])
    if result["stderr"]:
        lines.append("stderr/traceback:\n" + result["stderr"])
    if result["success"]:
        lines.append("[No error -- execution succeeded]")
    return "\n\n".join(lines)

## 6. Trajectory logger

Every (thought, code, observation, reflection) turn across every task gets appended here, independent of the LLM thread itself, so it's easy to render for the report.

In [ ]:
TRAJECTORY_LOG = []  # list of dicts, one per attempt across all tasks/sessions

def render_entry_md(entry):
    parts = [f"### Attempt {entry['attempt']}", "", "**Thought / Plan:**", "", entry["thought"], ""]
    if entry.get("code"):
        parts += ["**Action (code):**", "```python", entry["code"], "```", ""]
    parts += ["**Observation:**", "```", entry["observation"], "```", ""]
    return "\n".join(parts)

def render_reflection_md(reflection):
    return "\n".join(["**Reflection:**", "", reflection, "", "---", ""])

## 7. Agent session — the ReAct + Reflexion loop controller

A small state machine so the same loop logic drives both the headless notebook demo and the front-end:

- `status == "pending"` — ready for another plan+execute turn
- `step()` — one Thought -> Action -> Observation turn; on failure with attempts remaining, immediately runs the Reflexion turn in the same thread and goes back to `"pending"`; on failure with no attempts left, goes to `"stopped"`; on success, goes to `"awaiting_confirmation"`
- `confirm(accepted)` — the HITL checkpoint after a successful run; `True` -> `"succeeded"`, `False` -> back into the loop (as a Reflexion-worthy failure) if attempts remain, else `"stopped"`

This mirrors §5/§10 of the design doc: one continuous LLM thread per task, 3-attempt cap, HITL success check. Each session gets a `session_id` so the Flask backend in §10 can look it up across separate HTTP requests.

In [ ]:
import uuid

class AgentSession:
    def __init__(self, task, max_iterations=3):
        self.session_id = str(uuid.uuid4())
        self.task = task
        self.max_iterations = max_iterations
        self.attempt = 0
        self.status = "pending"  # pending | awaiting_confirmation | succeeded | stopped
        self.last_code = None
        self.last_output = None
        self.trajectory = []
        self.trajectory_md = f"## Task\n\n{task}\n\n---\n\n"
        self.messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f"Task: {task}"},
        ]

    def step(self):
        assert self.status == "pending", f"step() called while status={self.status!r}"
        self.attempt += 1

        plan_response = call_llm(self.messages)
        self.messages.append({"role": "assistant", "content": plan_response})

        thought = extract_thought(plan_response)
        code_str, extract_err = extract_code(plan_response)

        if code_str is None:
            observation = f"[Code extraction failed] {extract_err}"
            result = None
        else:
            result = run_in_sandbox(code_str)
            observation = format_observation(result)

        self.messages.append({"role": "user", "content": f"Observation:\n{observation}"})

        entry = {
            "task": self.task,
            "attempt": self.attempt,
            "thought": thought,
            "code": code_str,
            "observation": observation,
            "success": bool(result and result["success"]),
            "error_type": None if (result and result["success"]) else _classify_failure(code_str, result, extract_err),
            "reflection": None,
        }
        self.trajectory.append(entry)
        TRAJECTORY_LOG.append(entry)
        self.trajectory_md += render_entry_md(entry)

        if result is not None and result["success"]:
            self.status = "awaiting_confirmation"
            self.last_code = code_str
            self.last_output = result["stdout"]
        elif self.attempt >= self.max_iterations:
            self.status = "stopped"
        else:
            reflection = call_llm(self.messages + [{"role": "user", "content": REFLECTION_PROMPT}])
            self.messages.append({"role": "user", "content": REFLECTION_PROMPT})
            self.messages.append({"role": "assistant", "content": reflection})
            entry["reflection"] = reflection
            self.trajectory_md += render_reflection_md(reflection)
            self.status = "pending"

        return entry

    def confirm(self, accepted):
        assert self.status == "awaiting_confirmation", f"confirm() called while status={self.status!r}"
        if accepted:
            self.status = "succeeded"
            self.trajectory_md += "**User confirmed: this solves the task.**\n\n---\n\n"
            return

        self.trajectory_md += "**User rejected: output does not solve the task.**\n\n"
        if self.attempt >= self.max_iterations:
            self.status = "stopped"
            self.trajectory_md += "---\n\n"
            return

        self.messages.append({"role": "user", "content": REJECTED_BY_USER_PROMPT})
        reflection = call_llm(self.messages)
        self.messages.append({"role": "assistant", "content": reflection})
        if self.trajectory:
            self.trajectory[-1]["reflection"] = reflection
            self.trajectory[-1]["success"] = False
            self.trajectory[-1]["error_type"] = "rejected_by_user"
        self.trajectory_md += render_reflection_md(reflection)
        self.status = "pending"

### 7a. Repeated-failure classification

Per §9 of the design doc: two failed attempts count as the *same error type* if they share both the exception class and the failing line. This is logged automatically per attempt rather than judged by eye, so `TRAJECTORY_LOG` supports the ReAct-only vs. ReAct+Reflexion ablation later.

In [ ]:
import re as _re

_TRACEBACK_LAST_LINE_RE = _re.compile(r"^(\w+(?:Error|Exception|Warning))\b", _re.MULTILINE)
_FAILING_LINE_RE = _re.compile(r'File "[^"]*", line (\d+)')

def _classify_failure(code_str, result, extract_err):
    """Returns a compact (exception_class, failing_line) tag used to detect repeated identical failures."""
    if extract_err is not None:
        return f"extraction_error:{extract_err.split(':')[0]}"
    if result is None:
        return "unknown_error"
    if result["timed_out"]:
        return "timeout"
    stderr = result["stderr"] or ""
    exc_matches = _TRACEBACK_LAST_LINE_RE.findall(stderr)
    exc_class = exc_matches[-1] if exc_matches else f"exit_code_{result['returncode']}"
    line_matches = _FAILING_LINE_RE.findall(stderr)
    failing_line = line_matches[-1] if line_matches else "?"
    return f"{exc_class}@line_{failing_line}"

## 8. Headless demo (runs in the notebook, HITL via `input()`)

Drives an `AgentSession` to completion, printing each Thought/Action/Observation/Reflection as it happens, and pausing for a real y/n confirmation once the code runs cleanly.

In [ ]:
def _drive(session):
    """Generator: advances the session one step() at a time while status == 'pending'."""
    while session.status == "pending":
        session.step()
        yield session

def run_headless(task, max_iterations=3):
    session = AgentSession(task, max_iterations=max_iterations)
    for _ in _drive(session):
        pass
    print(session.trajectory_md)

    while session.status == "awaiting_confirmation":
        print(f"--- Code ran without error. Output:\n{session.last_output}\n")
        answer = input("Does this correctly solve the task? [y/n]: ").strip().lower()
        session.confirm(answer.startswith("y"))
        for _ in _drive(session):
            pass
        print(session.trajectory_md)

    print(f"Final status: {session.status} (after {session.attempt} attempt(s))")
    return session

In [ ]:
# Try it: an "easy" task per the design doc's evaluation plan (simple data transformation)
demo_session = run_headless(
    "Write a function that takes a list of numbers and returns a new list with duplicates removed, "
    "preserving the original order. Print the result for [4, 5, 4, 2, 5, 1, 2]."
)

## 9. Trajectory report

Renders `TRAJECTORY_LOG` as a table for inspection/inclusion in the writeup.

In [ ]:
import pandas as pd

pd.set_option("display.max_colwidth", 80)
pd.DataFrame(TRAJECTORY_LOG)[["task", "attempt", "success", "error_type"]]

## 10. Front-end (vanilla HTML/CSS/JS)

The front-end is **not generated by this notebook** — it lives in the repo's `frontend/` folder (`index.html`, `style.css`, `app.js`). Section 10a makes that folder available to the Colab runtime; 10b serves it with Flask (two JSON endpoints, `/api/run` and `/api/confirm`, driving the same `AgentSession`/`_drive()` loop used above, each streaming newline-delimited JSON so the trajectory panel updates per attempt); 10c exposes it publicly through a Cloudflare quick tunnel.

### 10a. Get the `frontend/` folder into the runtime

Pick one:
- **Git clone** (recommended, and how your teammate will run it too): push this project to GitHub, then set `REPO_URL` below.
- **Zip upload**: leave `REPO_URL` empty and you'll get a file picker — upload a zip containing `index.html`, `style.css`, `app.js`.

If you're running the notebook from inside the repo (e.g. locally, not on Colab), it just finds `./frontend` and does nothing else.

In [ ]:
import os, subprocess, zipfile

REPO_URL = ""  # e.g. "https://github.com/<you>/GenAI_Lab2.git"  -- leave "" to upload a zip instead
CLONE_DIR = "/content/genai_lab2"

def _find_frontend():
    candidates = ["frontend", os.path.join(CLONE_DIR, "frontend"), "/content/_frontend_upload"]
    for base in candidates:
        for root, _dirs, files_ in os.walk(base) if os.path.isdir(base) else []:
            if "index.html" in files_ and "app.js" in files_:
                return os.path.abspath(root)
    return None

FRONTEND_DIR = _find_frontend()

if FRONTEND_DIR is None and REPO_URL:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, CLONE_DIR], check=True)
    FRONTEND_DIR = _find_frontend()

if FRONTEND_DIR is None:
    from google.colab import files  # type: ignore
    print("Upload a zip containing index.html, style.css, app.js:")
    uploaded = files.upload()
    os.makedirs("/content/_frontend_upload", exist_ok=True)
    with zipfile.ZipFile(next(iter(uploaded))) as z:
        z.extractall("/content/_frontend_upload")
    FRONTEND_DIR = _find_frontend()

assert FRONTEND_DIR and os.path.isfile(os.path.join(FRONTEND_DIR, "index.html")), \
    "Could not locate the frontend/ folder -- set REPO_URL or upload a valid zip."
print("Serving frontend from:", FRONTEND_DIR)

### 10b. Backend (Flask)

Bridges HTTP requests to the same `AgentSession` / `_drive()` loop used in the headless demo. `SESSIONS` is a simple in-memory dict keyed by `session_id` — fine for a single-user Colab demo, not for concurrent multi-user deployment (see limitations below).

In [ ]:
import json as _json
from flask import Flask, request, jsonify, Response, send_from_directory

SESSIONS = {}

app = Flask(__name__, static_folder=None)

def session_to_json(session):
    return {
        "session_id": session.session_id,
        "status": session.status,
        "attempt": session.attempt,
        "max_iterations": session.max_iterations,
        "status_text": render_status(session),
        "trajectory": session.trajectory,
        "last_output": session.last_output,
    }

def render_status(session):
    if session.status == "succeeded":
        return f"Succeeded in {session.attempt} attempt(s)."
    if session.status == "stopped":
        return f"Stopped after {session.attempt} attempt(s) -- task not solved within the {session.max_iterations}-attempt budget."
    if session.status == "awaiting_confirmation":
        return f"Attempt {session.attempt} of {session.max_iterations} -- code ran without error. Please review and confirm below."
    return f"Attempt {session.attempt} of {session.max_iterations} -- retrying after failure..."

@app.route("/api/run", methods=["POST"])
def api_run():
    data = request.get_json(force=True) or {}
    task = (data.get("task") or "").strip()
    if not task:
        return jsonify({"error": "task is required"}), 400

    session = AgentSession(task, max_iterations=3)
    SESSIONS[session.session_id] = session

    def generate():
        for _ in _drive(session):
            yield _json.dumps(session_to_json(session)) + "\n"
        yield _json.dumps(session_to_json(session)) + "\n"

    return Response(generate(), mimetype="application/x-ndjson")

@app.route("/api/confirm", methods=["POST"])
def api_confirm():
    data = request.get_json(force=True) or {}
    session = SESSIONS.get(data.get("session_id"))
    if session is None or session.status != "awaiting_confirmation":
        return jsonify({"error": "no active run awaiting confirmation"}), 400
    accepted = bool(data.get("accepted"))

    def generate():
        session.confirm(accepted)
        yield _json.dumps(session_to_json(session)) + "\n"
        for _ in _drive(session):
            yield _json.dumps(session_to_json(session)) + "\n"
        yield _json.dumps(session_to_json(session)) + "\n"

    return Response(generate(), mimetype="application/x-ndjson")

@app.route("/")
def index():
    return send_from_directory(FRONTEND_DIR, "index.html")

@app.route("/<path:filename>")
def static_files(filename):
    return send_from_directory(FRONTEND_DIR, filename)

### 10c. Launch + Cloudflare tunnel

Runs Flask in a background thread, then starts a **Cloudflare quick tunnel** (`cloudflared --url http://localhost:5000`) and prints the public `https://<random>.trycloudflare.com` URL. No Cloudflare account or login needed — the binary is downloaded on first run.

The URL is **public while the cell's process is alive** (anyone with the link can hit your agent) and dies when the runtime stops. Re-run this cell to get a fresh one.

In [ ]:
import threading, time, re, os, stat, subprocess, urllib.request

PORT = 5000
CLOUDFLARED = "./cloudflared"

def _run_flask():
    app.run(host="0.0.0.0", port=PORT, use_reloader=False)

threading.Thread(target=_run_flask, daemon=True).start()
time.sleep(1.5)  # let Flask bind before the tunnel points at it

if not os.path.exists(CLOUDFLARED):
    print("Downloading cloudflared...")
    urllib.request.urlretrieve(
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        CLOUDFLARED,
    )
    os.chmod(CLOUDFLARED, os.stat(CLOUDFLARED).st_mode | stat.S_IEXEC)

tunnel = subprocess.Popen(
    [CLOUDFLARED, "tunnel", "--url", f"http://localhost:{PORT}", "--no-autoupdate"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
)

public_url = None
_url_re = re.compile(r"https://[-a-z0-9]+\.trycloudflare\.com")

def _watch_tunnel():
    global public_url
    for line in tunnel.stdout:  # keep draining so the pipe never blocks cloudflared
        if public_url is None:
            m = _url_re.search(line)
            if m:
                public_url = m.group(0)

threading.Thread(target=_watch_tunnel, daemon=True).start()

for _ in range(40):
    if public_url:
        break
    time.sleep(1)

if public_url:
    print("\nAgent UI is live at:", public_url)
else:
    print("\nTunnel URL not detected yet -- give it a few more seconds, or check for a cloudflared error above.")

## Known limitations of this prototype

- **Sandbox is process-isolated, not network/syscall-isolated.** A determined adversarial script could still open a socket. Fine for trusted classroom use against a code-generation model; not sufficient for untrusted multi-tenant use without OS-level controls (containers/seccomp).
- **`TRAJECTORY_LOG` and `SESSIONS` are single in-memory globals** — fine for one Colab session/demo, not for concurrent multi-user deployment.
- **No automated correctness check** — matches the design doc's decision to keep success HITL-confirmed rather than guessing from "no traceback."
- **Streaming is per-attempt, not per-token** — the front-end updates as each attempt finishes (via NDJSON chunks over `fetch`), not while the model is still generating a single response.
- **The Cloudflare quick tunnel is public and unauthenticated** — anyone with the `trycloudflare.com` link can drive the agent (and thus the sandbox) while the cell runs. Fine for a short demo; add auth or shut the tunnel down otherwise.
- **Ablation harness (ReAct-only vs. ReAct+Reflexion, §9) is not built yet** — `error_type` classification is in place to support it, but running the actual comparison across the easy/medium/hard task set is the next step.